In [18]:
import pandas as pd
import numpy as np

In [19]:
orders = pd.read_csv("ML-Dataset.csv")

# Example forecast output
forecast = pd.read_csv("subsystem1_product_recommendations.csv")

In [20]:
# Aggregate recent orders
recent_orders = orders.groupby("ProductName")["OrderItemQuantity"].sum().reset_index()

# Availability proxy
availability = orders.groupby("ProductName")["TotalItemQuantity"].mean().reset_index()

# Merge signals
risk_df = forecast.merge(recent_orders, on="ProductName", how="left")
risk_df = risk_df.merge(availability, on="ProductName", how="left")

In [21]:
baseline = orders.groupby("ProductName")["OrderItemQuantity"].mean().reset_index()
baseline.columns = ["ProductName", "baseline_demand"]

risk_df = risk_df.merge(baseline, on="ProductName", how="left")

In [22]:
risk_df["demand_pressure"] = (
    (risk_df["category_adjusted_forecast"] - risk_df["baseline_demand"]) /
    (risk_df["baseline_demand"] + 1e-6)
)

risk_df["demand_pressure"] = risk_df["demand_pressure"].clip(0, 1)

In [23]:
print(forecast.columns)
print(risk_df.columns)

Index(['CategoryName', 'ProductName', 'product_share', 'product_qty',
       'category_qty', 'probability_up', 'growth_signal',
       'category_adjusted_forecast', 'base_reorder_qty', 'confidence_score',
       'forecast_score'],
      dtype='str')
Index(['CategoryName', 'ProductName', 'product_share', 'product_qty',
       'category_qty', 'probability_up', 'growth_signal',
       'category_adjusted_forecast', 'base_reorder_qty', 'confidence_score',
       'forecast_score', 'OrderItemQuantity', 'TotalItemQuantity',
       'baseline_demand', 'demand_pressure'],
      dtype='str')


In [24]:
risk_df["availability_risk"] = 1 - (
    risk_df["TotalItemQuantity"] /
    (risk_df["TotalItemQuantity"] + risk_df["OrderItemQuantity"] + 1)
)

risk_df["availability_risk"] = risk_df["availability_risk"].clip(0,1)

In [25]:
volatility = orders.groupby("ProductName")["OrderItemQuantity"].std().reset_index()
volatility.columns = ["ProductName", "demand_std"]
volatility["demand_std"] = volatility["demand_std"].fillna(0)

risk_df = risk_df.merge(volatility, on="ProductName", how="left")

risk_df["volatility_factor"] = (
    risk_df["demand_std"] / (risk_df["baseline_demand"] + 1)
)

risk_df["volatility_factor"] = risk_df["volatility_factor"].clip(0, 1)
risk_df["volatility_factor"] = risk_df["volatility_factor"].fillna(0)

In [26]:
status_counts = orders.groupby(["ProductName","Status"]).size().unstack(fill_value=0)

status_counts["ops_risk"] = (
    status_counts.get("Pending",0) + status_counts.get("Cancelled",0)
) / status_counts.sum(axis=1)

status_counts = status_counts.reset_index()[["ProductName","ops_risk"]]

risk_df = risk_df.merge(status_counts, on="ProductName", how="left")

In [27]:
# Fill all missing factors
risk_df["demand_pressure"] = risk_df["demand_pressure"].fillna(0)
risk_df["availability_risk"] = risk_df["availability_risk"].fillna(0)
risk_df["ops_risk"] = risk_df["ops_risk"].fillna(0)

In [28]:
w1 = 0.35
w2 = 0.35
w3 = 0.20
w4 = 0.10

risk_df["risk_score"] = (
    w1*risk_df["demand_pressure"] +
    w2*risk_df["availability_risk"] +
    w3*risk_df["volatility_factor"] +
    w4*risk_df["ops_risk"]
)

In [29]:
def classify_risk(score):

    if score >= 0.7:
        return "High"

    elif score >= 0.4:
        return "Medium"

    else:
        return "Low"

risk_df["risk_level"] = risk_df["risk_score"].apply(classify_risk)

In [30]:
def risk_drivers(row):
    drivers = []

    if row["demand_pressure"] > 0.5:
        drivers.append("High forecasted demand")
    if row["availability_risk"] > 0.5:
        drivers.append("Low availability proxy")
    if row["volatility_factor"] > 0.5:
        drivers.append("Unstable demand")
    if row["ops_risk"] > 0.3:
        drivers.append("Operational issues")

    return ", ".join(drivers) if drivers else "No major risk drivers"

risk_df["risk_level"] = risk_df["risk_score"].apply(classify_risk)
risk_df["risk_drivers"] = risk_df.apply(risk_drivers, axis=1)

In [31]:
risk_output = risk_df[[
    "ProductName",
    "risk_score",
    "risk_level",
    "risk_drivers"
]]

risk_output.head()

,ProductName,risk_score,risk_level,risk_drivers
0,Crucial CT525MX300SSD1,0.587267,Medium,High forecasted demand
1,Corsair Dominator Platinum,0.761514,High,"High forecasted demand, Low availability proxy"
2,Hitachi HUA723020ALA640,0.379695,Low,High forecasted demand
3,PNY SSD9SC240GMDA-RB,0.407917,Medium,"Low availability proxy, Operational issues"
4,Samsung MZ-V6P2T0BW,0.474804,Medium,"High forecasted demand, Low availability proxy"


In [32]:

risk_output.to_csv("risk_scoring_output.csv", index=False)

In [33]:
print(risk_df[[
    "ProductName",
    "demand_pressure",
    "availability_risk",
    "volatility_factor",
    "ops_risk"
]].head(10))

                  ProductName  demand_pressure  availability_risk  \
0      Crucial CT525MX300SSD1         1.000000           0.404389   
1  Corsair Dominator Platinum         1.000000           0.890717   
2     Hitachi HUA723020ALA640         0.726420           0.358423   
3        PNY SSD9SC240GMDA-RB         0.304699           0.632212   
4         Samsung MZ-V6P2T0BW         0.560873           0.594595   
5       Supermicro MBD-X10DAX         0.656738           0.285714   
6           Asus X99-E-10G WS         1.000000           0.295699   
7             ASRock C2750D4I         0.000000           0.390671   
8        ASRock X99 Extreme11         0.233349           0.348659   
9                Intel DG43RK         0.004538           0.573643   

   volatility_factor  ops_risk  
0           0.478657       0.0  
1           0.348817       0.3  
2           0.000000       0.0  
3           0.149992       0.5  
4           0.351954       0.0  
5           0.000000       0.0  
6        